# Zero-Shot Classification


## Viability test of the classification model


In [1]:
import os
import pypdf
from transformers import pipeline

In [2]:
# 1. Extract text from the (resume) PDF
def extract_text_from_pdf(ruta_pdf):
    reader = pypdf.PdfReader(ruta_pdf)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    return text

In [ ]:
DATA_PATH = os.path.join(os.getcwd(), "data")
MODEL_PATH = os.path.join(os.getcwd(), "model")

os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)

MODEL_NAME_US = "facebook/bart-large-mnli"
MODEL_NAME_ES = "Recognai/zeroshot_selectra_medium"
MODEL_TASK = "zero-shot-classification"
print(f"DATA_PATH: {DATA_PATH}")
print(f"MODEL_PATH: {MODEL_PATH}")

DATA_PATH: /app/classifier/data
MODEL_PATH: /app/classifier/model


In [4]:
resume_text_data = extract_text_from_pdf(
    os.path.join(DATA_PATH, "CV JOSU ABAD - English.pdf")
)
print(resume_text_data[:100])

LinkedIn
 +34 635900449
 josuabaduc@gmail.com
JOSU ABAD OTAÑO
Summary
Computer Engineering graduate 


In [5]:
# 2. Load the zero-shot classification pipeline in Spanish
classifier = pipeline(MODEL_TASK, model=MODEL_NAME_US)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [6]:
# 3. Define tags and the context of the job offer
labels = [
    "suitable for the position",
    "possibly suitable",
    "not suitable for the position",
]
job_description = "We are looking for a Python developer with 3 years of experience in Django and SQL."

# We combine the job offer and the CV as context for the model.
prompt = f"Job offer: {job_description}\n\nCandidate profile: {resume_text_data}"

In [7]:
# 4. Get prediction
result = classifier(
    prompt, candidate_labels=labels, hypothesis_template="This candidate is {}."
)

print(f"Main result: {result['labels'][0]}")
print(f"Confidence: {result['scores'][0]:.2%}")

Main result: possibly suitable
Confidence: 55.78%


## Storing the model for fast future inference


In [8]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

In [9]:
# Upload from Hugging Face Hub
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_US)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME_US)

# Save to your hard drive
tokenizer.save_pretrained(MODEL_PATH)
model.save_pretrained(MODEL_PATH)

print(f"Model successfully saved in: {MODEL_PATH}")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model successfully saved in: /app/classifier/model
